1: Đọc dữ liệu gốc cục bộ

In [1]:
import os
import re
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Tự động tìm thư mục gốc dự án
cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in [cwd, *cwd.parents] if (p / "data" / "raw").is_dir()),
    cwd,
)
csv_path = PROJECT_DIR / "data" / "raw" / "combined_data.csv"
df = pd.read_csv(csv_path)

print(f"Đường dẫn dữ liệu: {csv_path}")
print(f"Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")

Đường dẫn dữ liệu: D:\spam-email-classifier\data\raw\combined_data.csv
Kích thước dữ liệu gốc: 83,448 dòng, 2 cột


2: Làm sạch dữ liệu

In [2]:
# 1. Xóa giá trị thiếu (nếu có)
df_clean = df.dropna(subset=['text', 'label']).copy()

# 2. Kiểm tra và xóa toàn bộ các dòng có nội dung trùng lặp nhưng khác nhãn (keep=False)
dup_mask = df_clean.duplicated(subset=['text'], keep=False)
print(f"Số dòng có nội dung trùng lặp cần loại bỏ: {dup_mask.sum()}")
df_clean = df_clean[~dup_mask]

# 3. Chuẩn hóa văn bản cơ bản (chuyển chữ thường, gom nhiều khoảng trắng thành 1)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['text'] = df_clean['text'].apply(clean_text)

# 4. Loại bỏ các dòng rỗng sau khi làm sạch (nếu có)
df_clean = df_clean[df_clean['text'] != ''].reset_index(drop=True)

print(f"Kích thước sau khi làm sạch: {df_clean.shape[0]:,} dòng")

Số dòng có nội dung trùng lặp cần loại bỏ: 4
Kích thước sau khi làm sạch: 83,444 dòng


3: Chia tập Train (80%) và Test (20%)

In [3]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)

print(f"Tập Train: {train_df.shape[0]:,} dòng")
print(f"Tập Test : {test_df.shape[0]:,} dòng")

print("\nTỷ lệ nhãn tập Train (%):")
print((train_df['label'].value_counts(normalize=True) * 100).round(2))

print("\nTỷ lệ nhãn tập Test (%):")
print((test_df['label'].value_counts(normalize=True) * 100).round(2))

Tập Train: 66,755 dòng
Tập Test : 16,689 dòng

Tỷ lệ nhãn tập Train (%):
label
1    52.62
0    47.38
Name: proportion, dtype: float64

Tỷ lệ nhãn tập Test (%):
label
1    52.62
0    47.38
Name: proportion, dtype: float64


4: Lưu tập train và test vào data/processed

In [4]:
output_dir = PROJECT_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.csv"
test_path = output_dir / "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Đã lưu tập train tại: {train_path}")
print(f"Đã lưu tập test  tại: {test_path}")

Đã lưu tập train tại: D:\spam-email-classifier\data\processed\train.csv
Đã lưu tập test  tại: D:\spam-email-classifier\data\processed\test.csv
